# Métodos de reducción de dimensionalidad
## Análisis de correlación canónica (ACC)

In [2]:
# Análisis de correlación canónica (ACC)

import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()

# Datos: Índice de Capacidad Estadística
# Territorial (ICET) 2023.
# https://www.dane.gov.co/files/operaciones/ICET/anex-ICET-2023.xlsx

# Por departamento
# Análisis de correlación canónica (ACC)
# Datos: Índice de Capacidad Estadística
# Territorial (ICET) 2023.
# https://www.dane.gov.co/files/operaciones/ICET/anex-ICET-2023.xlsx

# Por departamento
icet_dpto = (
    pd.read_excel(
        "anex-ICET-2023.xlsx",
        sheet_name="ICET_C1_Dptos",
        usecols="A:Q",
        skiprows=7,
        nrows=32
    )
)

icet_dpto.iloc[:, 0] = icet_dpto.iloc[:, 0].astype(str).str[:13]

icet_dpto = (
    icet_dpto
    .set_index(icet_dpto.columns[0])
    .drop(columns=icet_dpto.columns[1])
    .rename(columns={
        icet_dpto.columns[2]: "ICET",
        "Entorno Institucional": "EI",
        "Infraestructura": "INF",
        "Metodología estadística": "ME",
        "Accesibilidad y uso": "AU",
        "Marco institucional": "MI",
        "Gestión del conocimiento": "GC",
        "Percepción suficiencia de recursos humanos, tecnológicos y financieros": "PSRH",
        "Uso de herramientas de procesamiento": "UHP",
        "Formulación de indicadores": "FI",
        "Aprovechamiento de Registros Administrativos": "ARA",
        "Operaciones estadísticas": "OE",
        "Disponibilidad": "Disp",
        "Accesibilidad": "Acc",
        "Uso": "Uso"
    })
)

Saving anex-ICET-2023.xlsx to anex-ICET-2023.xlsx


In [4]:
# Por municipios
icet_ciudades = pd.read_excel(
    "anex-ICET-2023.xlsx",
    sheet_name="ICET_C2_Ciudades",
    usecols="A:Q",
    skiprows=7,
    nrows=20
)

icet_pdet = pd.read_excel(
    "anex-ICET-2023.xlsx",
    sheet_name="ICET_C3_PDET",
    usecols="A:Q",
    skiprows=7,
    nrows=165
)

icet_resto = pd.read_excel(
    "anex-ICET-2023.xlsx",
    sheet_name="ICET_C4_Resto_país",
    usecols="A:Q",
    skiprows=7,
    nrows=896
)

icet_munic = pd.concat([icet_ciudades, icet_pdet, icet_resto], ignore_index=True)

icet_munic = (
    icet_munic
    .iloc[:, 2:]
    .rename(columns={
        icet_munic.columns[2]: "ICET",
        "Entorno Institucional": "EI",
        "Infraestructura": "INF",
        "Metodología estadística": "ME",
        "Accesibilidad y uso": "AU",
        "Marco institucional": "MI",
        "Gestión del conocimiento": "GC",
        "Percepción suficiencia de recursos humanos, tecnológicos y financieros": "PSRH",
        "Uso de herramientas de procesamiento": "UHP",
        "Formulación de indicadores": "FI",
        "Aprovechamiento de Registros Administrativos": "ARA",
        "Operaciones estadísticas": "OE",
        "Disponibilidad": "Disp",
        "Accesibilidad": "Acc",
        "Uso": "Uso"
    })
)

In [5]:
# Estandarización
# Para departamentos
icet_z = icet_dpto.copy()
cols_num = icet_z.select_dtypes(include=np.number).columns
icet_z[cols_num] = icet_z[cols_num].apply(lambda x: (x - x.mean()) / x.std(ddof=1))

# Para municipios (quitar #)
# icet_z = icet_munic.copy()
# cols_num = icet_z.select_dtypes(include=np.number).columns
# icet_z[cols_num] = icet_z[cols_num].apply(lambda x: (x - x.mean()) / x.std(ddof=1))

In [6]:
# Grupos de variables
X = icet_z[["MI", "GC", "PSRH", "UHP"]].copy()

Y = icet_z[["FI", "ARA", "OE", "Disp", "Acc", "Uso"]].copy()

In [7]:
# Análisis de correlación canónica
from statsmodels.multivariate.cancorr import CanCorr

acc = CanCorr(Y, X)
acc.cancorr  # correlaciones canónicas

array([0.90514109, 0.61688321, 0.32627408, 0.14927088])

In [8]:
# Matriz de coeficientes canónicos (¿>0.3?)
xcoef = pd.DataFrame(
    acc.x_cancoef,
    index=X.columns,
    columns=[f"Dim{i}" for i in range(1, acc.x_cancoef.shape[1] + 1)]
)
display(xcoef)  # para X

ycoef = pd.DataFrame(
    acc.y_cancoef,
    index=Y.columns,
    columns=[f"Dim{i}" for i in range(1, acc.y_cancoef.shape[1] + 1)]
)
display(ycoef)  # para Y

,Dim1,Dim2,Dim3,Dim4
MI,0.111483,-0.053491,0.136073,0.190532
GC,0.019240,0.203332,-0.034889,-0.091593
PSRH,-0.004282,-0.082350,0.091440,-0.222015
UHP,0.088102,-0.027584,-0.205208,0.047733


,Dim1,Dim2,Dim3,Dim4
FI,0.003374,0.113870,0.151240,0.081448
ARA,0.066103,-0.181552,0.061856,0.106935
OE,0.095960,-0.075278,-0.035237,-0.106877
Disp,-0.020204,0.076805,-0.030995,0.167627
Acc,0.046968,0.078581,-0.029052,-0.238721
Uso,0.010327,0.109806,-0.112875,0.093145


In [9]:
# Pruebas de significancia de los coeficientes de correlación canónica
pval = acc.corr_test().stats["Pr > F"]
display(pval.round(3))

,Pr > F
0,0.000422
1,0.497619
2,0.923245
3,0.912404
